# 01 — เล่นกับ Prithvi-EO-2.0-300M (MAE reconstruction) บน CPU

**GISGPT** · ดูว่า Geospatial Foundation Model "เห็น" ภาพดาวเทียมยังไง

Prithvi-EO-2.0 ฝึกด้วยวิธี **MAE (Masked Autoencoder)** — โค้ดจะ **ปิดบัง 75%** ของภาพ
แล้วให้โมเดลเดา/สร้างส่วนที่หายไปใหม่ ถ้าโมเดลเดาได้ใกล้เคียง = โมเดลเข้าใจโครงสร้างของพื้นที่นั้นจริง

รันบน **CPU เครื่องเราได้เลย** (~1–2 นาที/ชุด 4 เฟรม ข้อมูล Mexico จาก repo ของ IBM)

> ⚠️ ต้องดาวน์โหลด checkpoint ก่อน (1.33GB):
> `hf download ibm-nasa-geospatial/Prithvi-EO-2.0-300M Prithvi_EO_V2_300M.pt --local-dir models/pretrained`

In [ ]:
import json, os

cfg = json.load(open('models/pretrained/config.json', encoding='utf-8'))['pretrained_cfg']
print('แบนด์ที่ใช้ฝึก:', cfg['bands'])          # B02-B07 = น้ำเงิน→เรดเอจ
print('ขนาดภาพ:', cfg['img_size'], '| patch:', cfg['patch_size'], '| depth:', cfg['depth'])
print('embed_dim:', cfg['embed_dim'], '| heads:', cfg['num_heads'])
print('mean (z-score):', cfg['mean'])
print('std  (z-score):', cfg['std'])
print('จำนวนพารามิเตอร์โมเดลไฟล์:', f"{os.path.getsize('models/pretrained/Prithvi_EO_V2_300M.pt')/1e9:.2f} GB")

## รัน MAE reconstruction

ใช้ `inference.py` ต้นฉบับของ IBM (โหลดมาจาก repo พร้อม checkpoint) กับภาพ HLS 4 ช่วงเวลา
ของพื้นที่เดียวกัน (Mexico, T13REM) — โมเดลฝึกมาด้วย 4 time steps

In [ ]:
import subprocess, sys, time

files = [f'data/sample/Mexico_HLS.S30.T13REM.{d}.v2.0_cropped.tif' for d in [
    '2018026T173609', '2018106T172859', '2018201T172901', '2018266T173029']]

t0 = time.time()
r = subprocess.run([
    sys.executable, 'models/pretrained/inference.py',
    '--data_files', *files,
    '--config_path', 'models/pretrained/config.json',
    '--checkpoint', 'models/pretrained/Prithvi_EO_V2_300M.pt',
    '--output_dir', 'outputs/mae_demo', '--rgb_outputs',
], capture_output=True, text=True, encoding='utf-8', errors='replace')
print(r.stdout[-800:])
print(f'⏱️ ใช้เวลา: {(time.time()-t0)/60:.1f} นาที')
assert r.returncode == 0, r.stderr[-500:]

## ดูผลลัพธ์

แต่ละเฟรมมี 4 แผง: **ต้นฉบับ | โมเดลสร้างใหม่ | mask (จุดดำ = ส่วนที่ถูกปิด) | ซ้อนทับ**

In [ ]:
import rasterio, numpy as np
from PIL import Image

def load_rgb(p):
    with rasterio.open(p) as src:
        return np.moveaxis(src.read(), 0, -1)

def norm(x):
    x = x.astype('float32')
    lo, hi = np.percentile(x, 2), np.percentile(x, 98)
    return np.clip((x-lo)/(hi-lo+1e-6), 0, 1)

for t in range(4):
    o, r, m = (norm(load_rgb(f'outputs/mae_demo/{k}_t{t}.tiff')) for k in ['original_rgb', 'predicted_rgb', 'masked_rgb'])
    H, W = o.shape[:2]
    canvas = np.zeros((H, W*4, 3), dtype='uint8')
    for i, im in enumerate([o, r, m, (m*0.5 + o*0.5)]):
        canvas[:, i*W:(i+1)*W] = (im*255).astype('uint8')
    Image.fromarray(canvas).save(f'outputs/mae_demo/preview_t{t}.png')
print('✅ preview ครบ 4 เฟรม — เปิดดูที่ outputs/mae_demo/preview_t*.png')

## สิ่งที่ได้เรียนรู้

1. **แบนด์ของโมเดล = B02–B07** (น้ำเงิน→เรดเอจ 3 ตัว) — ไม่ใช่ NIR/SWIR
2. **preprocessing = z-score** ด้วย mean/std จาก config (`(x-mean)/std`) — ต่างจาก /10000 ของ Sen4Map
3. โมเดลรับ **4 time steps** (ฝึกมาแบบนั้น) — ภาพเดียวก็พอได้แต่ผลอาจเพี้ยน
4. CPU เครื่องเราใช้เวลา ~1–2 นาที/รอบ — Colab GPU จะเร็วกว่าร้อยเท่า

ขั้นต่อไป: `04-finetune-landcover-colab.ipynb` — เอา backbone นี้ไป fine-tune เป็น land cover
แล้ว export ONNX กลับมาใช้ในแอป GISGPT